In [ ]:
"""
MVTec Bottle Quality Controller — Supervised Binary Anomaly Classifier
======================================================================
Production-grade training pipeline for supervised binary classification
of MVTec AD bottle images as 'good' or 'anomaly'.

Architecture:
    - Backbone  : ResNet18, pretrained on ImageNet (frozen)
    - Head      : Custom nn.Linear binary classifier
    - Optimizer : AdamW with weight decay
    - Precision : AMP (FP16 autocast + GradScaler)
    - Sampling  : WeightedRandomSampler for class imbalance

Dataset Engineering:
    MVTec AD is originally designed for unsupervised anomaly detection.
    This pipeline restructures it into a supervised binary classification
    problem by aggregating all defect sub-folders into a unified 'anomaly'
    class — enabling direct comparison between supervised and unsupervised
    paradigms on identical data.

Deployment:
    FastAPI + Docker (CPU inference)

Author  : Alejandro Toro Arrabal
Version : 2.0.0
"""

# ── Standard Library
import copy
import logging
import math
import os
import random
import shutil
import sys
import tarfile
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Optional
from urllib.request import urlretrieve

# ── Third Party
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import recall_score, roc_auc_score
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms
from tqdm import tqdm

In [ ]:
# ════════════════════════════════════════════════════════
# 1. LOGGING INFRASTRUCTURE
# ════════════════════════════════════════════════════════

def setup_logger(name: str = "Factory_MVTec") -> logging.Logger:
    """Configures a structured stdout logger with duplicate handler prevention."""

    logger = logging.getLogger(name)

    if logger.hasHandlers():
        logger.handlers.clear()

    logger.setLevel(logging.INFO)
    logger.propagate = False  # Prevents Colab's hidden logger from duplicating output

    formatter = logging.Formatter(
        fmt="%(asctime)s - [%(levelname)s] - %(message)s",
        datefmt="%H:%M:%S"
    )
    
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger


logger = setup_logger()

In [ ]:
# ════════════════════════════════════════════════════════
# 2. CONFIGURATION
# ════════════════════════════════════════════════════════

@dataclass(frozen= True)
class InspectionConfig:
    """
    Immutable configuration for the MVTec Bottle Quality Control system.

    Frozen dataclass prevents accidental mutation during training lifecycle.
    All hyperparameters and paths are centralized here — no magic numbers
    scattered through the codebase.
    """

    # ── Data Sources
    DATA_URL: str = "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420937370-1629958698/bottle.tar.xz"
    ARCHIVE_NAME: str = "bottle.tar.xz"
    DATA_ROOT: Path = Path("bottle")

    # ── Dataset Engineering
    # Fraction of defective test images moved into the training anomaly set.
    # 20% → train anomaly, 80% → val anomaly. Majority retained for validation.
    TRAIN_SPLIT_RATIO: float = 0.20

    # ── Training Hyperparameters
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 2
    NUM_CLASSES: int = 2
    EPOCHS: int = 20
    LEARNING_RATE: float = 1e-4
    WEIGHT_DECAY: float = 1e-4   # L2 regularization — prevents head weights from growing large
    SEED: int = 42

    # ── Checkpointing
    SAVE_PATH: str = "best_model_mvtec.pth"

In [ ]:
# ════════════════════════════════════════════════════════
# 3. BOTTLE QUALITY CONTROLLER — OOP SYSTEM
# ════════════════════════════════════════════════════════

class BottleQualityController:
    """
    End-to-end supervised anomaly classification pipeline for MVTec bottles.

    Responsibilities:
        - Dataset download, extraction, and binary restructuring
        - DataLoader construction with class balancing via WeightedRandomSampler
        - ResNet18 fine-tuning with frozen backbone and AMP training
        - Recall-primary checkpointing (defect recall > accuracy for industrial AOI)
        - AUROC evaluation for anomaly detection benchmarking
        - API-ready inference (accepts numpy arrays, returns confidence scores)
        - Visual inspection grid for qualitative analysis

    Usage:
        config = InspectionConfig(EPOCHS=20)
        system = BottleQualityController(config)
        system.ingest_and_structure_data()
        system.prepare_dataloaders()
        system.train()
    """

    def __init__(self, config: InspectionConfig) -> None:
        self.cfg = config
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # State placeholders
        self.model: Optional[nn.Module] = None
        self.dataloaders: Dict = {}
        self.dataset_sizes: Dict = {}
        self.class_names: list = []
        self.anomaly_idx: Optional[int] = None
        self.scaler = None

        self._set_seed()
        self._audit_hardware()

        # ── Training transforms — stochastic augmentation for generalization
        self.train_transform = transforms.Compose([
            transforms.Resize((224, 224)),

            # Industrial augmentations — conveyor belt simulation
            # RandomAffine simulates vibrations and misalignments on the production line
            # degrees=5     : slight rotation variance
            # translate=0.05: slight position shift
            # scale=0.98-1.02: slight camera zoom variance
            transforms.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.98, 1.02)),

            # Sensor noise simulation — robustness against factory lighting changes
            transforms.ColorJitter(brightness=0.1, contrast=0.1),

            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        # ── Validation transforms — deterministic, must match predict_image exactly
        # CRITICAL: any change here must be mirrored in predict_image()
        self.val_transform = transforms.Compose([
            transforms.Resize((224, 224)),    # direct resize — no crop
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    # ────────────────────────────────────────────────────
    # PRIVATE UTILITIES
    # ────────────────────────────────────────────────────

    def _set_seed(self) -> None:
        """
        Locks all random number generators for full deterministic execution.

        Required components:
            PYTHONHASHSEED          : Python dict hash randomization
            random                  : Python stdlib random
            numpy                   : NumPy operations
            torch.manual_seed       : CPU tensor ops
            torch.cuda.manual_seed_all : All GPU tensor ops
            cudnn.deterministic     : Deterministic CUDA kernels
            cudnn.benchmark = False : Disables auto-tuner (non-deterministic)

        Note: ~10-20% training speed penalty. Re-enable benchmark=True
        in production inference for maximum throughput.
        """
        seed = self.cfg.SEED
        os.environ["PYTHONHASHSEED"] = str(seed)
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        if torch.cuda.is_available():
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        logger.info(f"Deterministic seed locked: {seed}")

    def _audit_hardware(self) -> None:
        """
        Logs hardware capabilities for dynamic resource planning.

        In production, knowing GPU VRAM enables dynamic batch sizing
        and prevents OOM errors before they occur.
        """
        logger.info(f"Computation Device: {self.device.type.upper()}")
        if self.device.type == "cuda":
            gpu_name = torch.cuda.get_device_name(0)
            gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
            logger.info(f"   GPU: {gpu_name} | VRAM: {gpu_mem:.2f}GB")

    def _download_with_progress(self, url: str, filename: str) -> None:
        """Downloads a file with tqdm progress bar."""
        with tqdm(unit="B", unit_scale=True, unit_divisor=1024, desc=filename) as t:
            def reporthook(block_num, block_size, total_size):
                if total_size > 0:
                    t.total = total_size
                t.update(block_size)
            urlretrieve(url, filename, reporthook=reporthook)

    # ────────────────────────────────────────────────────
    # PUBLIC PIPELINE METHODS
    # ────────────────────────────────────────────────────

    def ingest_and_structure_data(self) -> None:
        """
        Downloads, extracts, and restructures the MVTec AD bottle dataset
        for supervised binary classification.

        MVTec AD original structure (unsupervised):
            train/good
            test/good, test/broken_large, test/contamination, ...

        Restructured binary structure (supervised):
            train/good     vs  train/anomaly   (TRAIN_SPLIT_RATIO of defects)
            test/good      vs  test/anomaly    (remaining defects)

        Engineering decision:
            Aggregating all defect sub-folders into a unified 'anomaly' class
            enables supervised training without leakage between splits.
            Each defect image is prefixed with its defect type name to maintain
            traceability back to the original MVTec category.
        """
        root = self.cfg.DATA_ROOT

        # ── Download & Extract
        if not root.exists():
            logger.info("Downloading MVTec Bottle Dataset (~200MB)...")
            self._download_with_progress(self.cfg.DATA_URL, self.cfg.ARCHIVE_NAME)
            with tarfile.open(self.cfg.ARCHIVE_NAME, "r:xz") as tar:
                tar.extractall()
            os.remove(self.cfg.ARCHIVE_NAME)
            logger.info("Dataset extracted.")
        else:
            logger.info("Dataset found. Skipping download.")

        # ── Restructure to binary classification
        train_anomaly_dir = root / "train" / "anomaly"
        val_anomaly_dir = root / "test" / "anomaly"

        if train_anomaly_dir.exists() and val_anomaly_dir.exists():
            logger.info("Binary structure already enforced. Skipping restructuring.")
            return

        logger.info("Executing binary restructuring strategy...")
        os.makedirs(train_anomaly_dir, exist_ok=True)
        os.makedirs(val_anomaly_dir, exist_ok=True)

        test_dir = root / "test"
        defect_types = [
            d.name for d in test_dir.iterdir()
            if d.is_dir() and d.name not in {"good", "anomaly"}
        ]

        for defect in defect_types:
            source_path = test_dir / defect
            images = sorted([
                f for f in source_path.iterdir()
                if f.suffix.lower() in {".png", ".jpg"}
            ])
            n_train = math.ceil(len(images) * self.cfg.TRAIN_SPLIT_RATIO)

            for i, img_path in enumerate(images):
                # Prefix with defect type for full traceability
                new_name = f"{defect}_{img_path.name}"
                target_dir = train_anomaly_dir if i < n_train else val_anomaly_dir
                shutil.move(str(img_path), str(target_dir / new_name))

            try:
                os.rmdir(source_path)
            except OSError:
                pass

        logger.info(
            f"Restructuring complete. Classes: good vs anomaly | "
            f"Defect types aggregated: {defect_types}"
        )

    def prepare_dataloaders(self) -> None:
        """
        Constructs optimized DataLoaders with class balancing for the
        heavily imbalanced MVTec binary classification problem.

        Class Balancing — WeightedRandomSampler:
            Industrial datasets have ~10:1 nominal-to-defect ratio.
            Naive training causes the model to always predict 'good'.
            WeightedRandomSampler assigns inverse-frequency weights per sample,
            forcing balanced mini-batches regardless of natural class distribution.
            samples with replacement=True — minority class images appear multiple
            times per epoch.

        DataLoader Optimizations:
            pin_memory=True       : Locks tensors in non-pageable RAM for direct
                                    DMA transfer across PCIe — bypasses CPU bottleneck
            prefetch_factor=2     : CPU pre-loads 2 batches ahead of GPU processing
                                    — eliminates GPU starvation idle time
            persistent_workers=True: Prevents OS from destroying/rebuilding worker
                                     processes between epochs — eliminates spawn overhead
        """
        logger.info("Building optimized data pipelines...")

        train_ds = datasets.ImageFolder(
            str(self.cfg.DATA_ROOT / "train"), self.train_transform
        )
        val_ds = datasets.ImageFolder(
            str(self.cfg.DATA_ROOT / "test"), self.val_transform
        )

        # ── WeightedRandomSampler — inverse frequency class balancing
        targets = torch.tensor(train_ds.targets)
        class_weights = 1.0 / torch.bincount(targets).float()
        sample_weights = class_weights[targets]

        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True  # minority class images resampled multiple times per epoch
        )

        self.dataloaders = {
            # shuffle=False — sampler controls stochastic selection
            "train": DataLoader(
                train_ds,
                batch_size=self.cfg.BATCH_SIZE,
                sampler=sampler,
                num_workers=self.cfg.NUM_WORKERS,
                pin_memory=True,
                prefetch_factor=2,
                persistent_workers=True,
            ),
            # shuffle=False — deterministic order for consistent val metrics
            "val": DataLoader(
                val_ds,
                batch_size=self.cfg.BATCH_SIZE,
                shuffle=False,
                num_workers=self.cfg.NUM_WORKERS,
                pin_memory=True,
                prefetch_factor=2,
                persistent_workers=True,
            ),
        }

        self.dataset_sizes = {"train": len(train_ds), "val": len(val_ds)}
        self.class_names = train_ds.classes
        self.anomaly_idx = train_ds.class_to_idx["anomaly"]

        logger.info(
            f"Pipelines ready | Classes: {self.class_names} | "
            f"Train: {self.dataset_sizes['train']} | Val: {self.dataset_sizes['val']}"
        )

    def build_model(self) -> None:
        """
        Initializes ResNet18 with frozen backbone for transfer learning.

        Architecture decision:
            - Backbone frozen: ImageNet features are already robust for
              detecting surface anomalies. Only the classification head
              needs to learn the good/anomaly decision boundary.
            - AdamW optimizer: decouples weight decay from gradient scaling
              (unlike standard Adam which corrupts regularization).
            - GradScaler: required companion to autocast. Scales loss before
              backward() to prevent FP16 gradient underflow to zero.
        """
        logger.info("Initializing ResNet18 backbone...")

        is_gpu = torch.cuda.is_available()
        self.scaler = torch.amp.GradScaler("cuda" if is_gpu else "cpu", enabled=is_gpu)

        # Load pretrained backbone
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        self.model = models.resnet18(weights=weights)

        # Freeze feature extractor — backbone learns nothing, only head updates
        for param in self.model.parameters():
            param.requires_grad = False

        # Replace classification head for binary AOI task
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, self.cfg.NUM_CLASSES)
        self.model = self.model.to(self.device)

        # AdamW — correct weight decay decoupled from gradient scaling
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.AdamW(
            self.model.fc.parameters(),
            lr=self.cfg.LEARNING_RATE,
            weight_decay=self.cfg.WEIGHT_DECAY,
        )
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", factor=0.1, patience=3
        )
        logger.info("Architecture setup complete.")

    def train(self) -> None:
        """
        Executes training loop with recall-primary checkpointing.

        Checkpointing strategy — Recall first, Accuracy as tie-breaker:
            For industrial defect detection, missing a defect is more
            expensive than a false alarm. Recall (catching all defects)
            is therefore the primary optimization target, with accuracy
            used only to break equal-recall ties.

        AMP strategy:
            autocast() casts eligible operations to FP16 during forward pass.
            GradScaler inflates loss before backward() to prevent FP16
            gradient underflow, then divides gradients back before optimizer step.
            Together they approximately halve VRAM consumption without
            meaningful accuracy loss.

        Metrics tracked per epoch:
            - Loss       : CrossEntropyLoss
            - Accuracy   : Overall correct predictions
            - Recall     : Specifically anomaly detection recall (pos_label=anomaly_idx)
            - AUROC      : Threshold-independent ranking quality (val only)
        """
        if self.model is None:
            self.build_model()

        since = time.time()
        best_model_wts = copy.deepcopy(self.model.state_dict())
        best_recall, best_acc = 0.0, 0.0

        logger.info(f"Starting training protocol | Epochs: {self.cfg.EPOCHS}")

        for epoch in range(self.cfg.EPOCHS):
            logger.info(f"--- Epoch {epoch + 1}/{self.cfg.EPOCHS} ---")

            for phase in ["train", "val"]:
                if phase == "train":
                    self.model.train()
                else:
                    self.model.eval()

                running_loss = 0.0
                running_corrects = 0
                all_preds: list = []
                all_labels: list = []
                all_probs: list = []  # for AUROC computation

                pbar = tqdm(self.dataloaders[phase], desc=phase.upper(), leave=False)

                for inputs, labels in pbar:
                    inputs = inputs.to(self.device)
                    labels = labels.to(self.device)
                    self.optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == "train"):
                        # AMP forward pass — eligible ops cast to FP16
                        with torch.amp.autocast(device_type=self.device.type):
                            outputs = self.model(inputs)
                            _, preds = torch.max(outputs, 1)
                            loss = self.criterion(outputs, labels)

                        if phase == "train":
                            # GradScaler prevents FP16 gradient underflow
                            self.scaler.scale(loss).backward()
                            self.scaler.step(self.optimizer)
                            self.scaler.update()

                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)

                    # Collect for epoch-level metrics
                    probs = torch.softmax(outputs.detach(), dim=1)[:, self.anomaly_idx]
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())
                    all_probs.extend(probs.cpu().numpy())

                    pbar.set_postfix(loss=f"{loss.item():.4f}")

                epoch_loss = running_loss / self.dataset_sizes[phase]
                epoch_acc = running_corrects.double() / self.dataset_sizes[phase]
                epoch_recall = recall_score(
                    all_labels, all_preds,
                    pos_label=self.anomaly_idx,
                    zero_division=0
                )

                log_str = (
                    f"{phase.upper()} | Loss: {epoch_loss:.4f} | "
                    f"Acc: {epoch_acc:.4f} | Anomaly Recall: {epoch_recall:.4f}"
                )

                if phase == "val":
                    # AUROC — threshold-independent anomaly ranking quality
                    try:
                        epoch_auroc = roc_auc_score(all_labels, all_probs)
                        log_str += f" | AUROC: {epoch_auroc:.4f}"
                    except ValueError:
                        pass  # only one class present in batch — skip

                    self.scheduler.step(epoch_loss)

                    # ── Recall-primary checkpointing
                    # Recall first — catching every defect is the primary objective
                    # Accuracy as tie-breaker — prefer accurate model when recall is equal
                    save_model = False
                    if epoch_recall > best_recall:
                        save_model, reason = True, "Recall Improvement"
                    elif epoch_recall == best_recall and epoch_acc > best_acc:
                        save_model, reason = True, "Accuracy Tie-Break"

                    if save_model:
                        best_recall = epoch_recall
                        best_acc = float(epoch_acc)
                        best_model_wts = copy.deepcopy(self.model.state_dict())
                        self.save_model(self.cfg.SAVE_PATH)
                        logger.info(f"New artifact saved ({reason})")

                logger.info(log_str)

        elapsed = time.time() - since
        logger.info(
            f"Training complete in {elapsed // 60:.0f}m {elapsed % 60:.0f}s | "
            f"Best Recall: {best_recall:.4f} | Best Accuracy: {best_acc:.4f}"
        )
        self.model.load_state_dict(best_model_wts)

    def load_model(self, path: str) -> None:
        """
        Loads the best model artifact and runs warmup inference.

        Warmup inference compiles CUDA kernels on first pass. Without it,
        the first real API request incurs a 3-5x latency spike.
        Uses a dummy zero array — no real image required.

        Raises:
            FileNotFoundError: If model artifact does not exist at path.
        """
        if not os.path.exists(path):
            raise FileNotFoundError(f"Model artifact not found: {path}")

        if self.model is None:
            self.build_model()

        self.model.load_state_dict(torch.load(path, map_location=self.device))
        logger.info(f"Artifact loaded from: {path}")

        # Warmup pass — compiles CUDA kernels before first real request
        logger.info("Running warmup inference...")
        dummy = torch.randn(1, 3, 224, 224).to(self.device)
        with torch.inference_mode():
            self.model(dummy)
        logger.info("Model warmed up and ready.")

    def predict_image(self, image: np.ndarray) -> Dict[str, Any]:
        """
        API-ready inference method. Accepts numpy arrays from FastAPI's
        in-memory byte decoding pipeline (cv2.imdecode output).

        Args:
            image: RGB numpy array of shape (H, W, 3), dtype uint8.
                   Note: OpenCV reads BGR — convert before passing:
                   image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        Returns:
            JSON-serializable dict:
            {
                "prediction": str,      # "good" or "anomaly"
                "confidence": float,    # probability of predicted class [0,1]
                "class_id": int         # 0 or 1
            }

        Raises:
            ValueError: If image is None or empty.
        """
        if image is None or image.size == 0:
            raise ValueError(
                "Invalid image — array is None or empty. "
                "Verify cv2.imdecode decoded the bytes correctly."
            )

        if self.model is None:
            raise RuntimeError("Model not loaded. Call load_model() before predict_image().")

        # ── Preprocessing — MUST match val_transform exactly
        # Any change to val_transform must be mirrored here.
        # Inconsistent preprocessing = silently degraded production accuracy.
        preprocess = transforms.Compose([
            transforms.Resize((224, 224)),    # matches val_transform
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        # Convert numpy array to PIL — the format torchvision transforms expect
        pil_image = Image.fromarray(image).convert("RGB")
        img_tensor = preprocess(pil_image).unsqueeze(0).to(self.device)

        self.model.eval()
        with torch.inference_mode():
            outputs = self.model(img_tensor)
            probs = torch.softmax(outputs, dim=1)
            confidence, pred_idx = probs.max(dim=1)

        return {
            "prediction" : self.class_names[int(pred_idx[0])],
            "confidence" : round(float(confidence[0]), 4),
            "class_id"   : int(pred_idx[0]),
        }

    def save_model(self, path: str) -> None:
        """Persists model state dict to disk."""
        torch.save(self.model.state_dict(), path)

    def smoke_test(self) -> bool:
        """
        Validates the full inference pipeline before deployment.

        Tests:
            - Model loads and executes a forward pass without error
            - Output tensor has the correct shape (1, NUM_CLASSES)
            - Softmax probabilities sum to 1.0

        Returns:
            True if all checks pass, False otherwise.
        """
        if self.model is None:
            logger.error("Smoke test failed: model not loaded.")
            return False

        try:
            dummy_input = torch.randn(1, 3, 224, 224).to(self.device)
            self.model.eval()

            with torch.inference_mode():
                output = self.model(dummy_input)
                probs = torch.softmax(output, dim=1)

            assert output.shape == (1, self.cfg.NUM_CLASSES), \
                f"Wrong output shape: {output.shape}"
            assert torch.allclose(probs.sum(), torch.tensor(1.0), atol=1e-5), \
                "Probabilities do not sum to 1"

            logger.info(
                f"Smoke test passed | Output shape: {output.shape} | "
                f"Probs sum: {probs.sum().item():.6f}"
            )
            return True

        except Exception as e:
            logger.error(f"Smoke test failed: {e}")
            return False

    def visualize_predictions(
        self, num_images: int = 6, save_path: str = "inspection_grid.png"
    ) -> None:
        """
        Qualitative analysis tool — generates visual inspection grid.

        Displays validation predictions vs ground truth with industrial
        color coding: green = correct classification, red = misclassification.
        Denormalizes tensors from ImageNet statistics for human-readable display.

        Args:
            num_images: Number of validation samples to visualize.
            save_path : Output path for the saved inspection grid.
        """
        if self.model is None:
            logger.error("Model not loaded. Cannot visualize.")
            return

        logger.info(f"Generating visual inspection grid ({num_images} samples)...")
        was_training = self.model.training
        self.model.eval()
        images_so_far = 0

        cols = 3
        rows = math.ceil(num_images / cols)
        plt.figure(figsize=(15, 5 * rows))

        with torch.inference_mode():
            for inputs, labels in self.dataloaders["val"]:
                inputs = inputs.to(self.device)
                labels = labels.to(self.device)
                outputs = self.model(inputs)
                _, preds = torch.max(outputs, 1)

                for j in range(inputs.size(0)):
                    images_so_far += 1
                    ax = plt.subplot(rows, cols, images_so_far)
                    ax.axis("off")

                    # Denormalize for human viewing — reverse ImageNet normalization
                    mean = np.array([0.485, 0.456, 0.406])
                    std = np.array([0.229, 0.224, 0.225])
                    img = inputs.cpu().data[j].numpy().transpose((1, 2, 0))
                    img = std * img + mean
                    img = np.clip(img, 0, 1)

                    # Industrial color coding
                    is_correct = preds[j] == labels[j]
                    color = "green" if is_correct else "red"
                    title = (
                        f"Pred: {self.class_names[preds[j]]}\n"
                        f"Real: {self.class_names[labels[j]]}"
                    )
                    ax.set_title(title, color=color, fontweight="bold")
                    ax.imshow(img)

                    if images_so_far == num_images:
                        self.model.train(mode=was_training)
                        plt.tight_layout()
                        plt.savefig(save_path, dpi=150, bbox_inches="tight")
                        logger.info(f"Inspection grid saved: {save_path}")
                        plt.show()
                        return

        self.model.train(mode=was_training)

In [ ]:
# ════════════════════════════════════════════════════════
# 4. EXECUTION ENTRYPOINT
# ════════════════════════════════════════════════════════

if __name__ == "__main__":

    # ── Configuration
    config = InspectionConfig(EPOCHS=20)

    # ── Phase 1: Training pipeline
    aoi_system = BottleQualityController(config)
    aoi_system.ingest_and_structure_data()
    aoi_system.prepare_dataloaders()
    aoi_system.train()

    # ── Phase 2: Production validation
    prod_system = BottleQualityController(config)
    prod_system.prepare_dataloaders()  # required to populate class_names mapping
    prod_system.load_model(config.SAVE_PATH)

    if prod_system.smoke_test():
        logger.info("[SYSTEM SECURED] Artifact validated. Ready for Docker integration.")
        # prod_system.visualize_predictions(num_images=6)
    else:
        logger.error("[SYSTEM FAILURE] Smoke test failed. Do not deploy.")